# 1. Kiểm tra GPU
Notebook này nên chạy trên Google Colab T4/A100.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.2f} GB / total: {total/1e9:.2f} GB')
else:
    print('NO GPU - chỉ nên dùng để setup hoặc test rất ngắn')


# 2. Mount Google Drive tùy chọn
Dùng nếu muốn cache model qua nhiều session.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'


# 3. Clone repo và cài dependencies


In [ ]:
%cd /content
!rm -rf Fast-VietTTS
!git clone https://github.com/baobui0709/Fast-VietTTS.git
%cd Fast-VietTTS

!pip install -q --upgrade pip
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchaudio==2.6.0
!pip install -q numpy==1.26.4 librosa soundfile num2words tqdm huggingface_hub gradio
!pip install -q s3tokenizer omegaconf conformer safetensors
!pip uninstall -y -q torchvision torchtext || true

%cd /content
!rm -rf TienHiep-TTS
!git clone https://github.com/yingchunbin/TienHiep-TTS.git
%cd /content/Fast-VietTTS


# 4. Import helper chung


In [ ]:
import sys, os, time, torch, torchaudio
from IPython.display import Audio, display

viterbox_src = '/content/TienHiep-TTS'
if viterbox_src not in sys.path:
    sys.path.insert(0, viterbox_src)

from src.engine import FastVietTTS
from src.audio_utils import prepare_reference, get_audio_duration


# 5. Load model


In [ ]:
tts = FastVietTTS(device='cuda' if torch.cuda.is_available() else 'cpu', use_fp16=False, compile_model=False)
print('Model loaded')


# 6. Upload hoặc tạo file .txt


In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    txt_path = next(iter(uploaded.keys()))
else:
    txt_path = '/content/sample_chapters.txt'
    open(txt_path, 'w', encoding='utf-8').write('Chương 1
Xin chào thế giới.

Chương 2
Đây là chương thứ hai.')
print('TXT:', txt_path)


# 7. Upload reference audio tùy chọn


In [ ]:
uploaded_ref = files.upload()
ref_path = None
if uploaded_ref:
    ref_path = next(iter(uploaded_ref.keys()))
    ref_path = prepare_reference(ref_path, output_path='/content/ref_batch.wav')
print('Reference:', ref_path)


# 8. Run batch generation


In [ ]:
from src.batch_generator import BatchGenerator
out_dir = '/content/batch_output'
gen = BatchGenerator(tts, out_dir)
t0 = time.time()
files_out = gen.generate_from_file(txt_path, reference_audio=ref_path, resume=True)
elapsed = time.time() - t0
print('Files:', files_out)
print(f'Total elapsed: {elapsed:.2f}s')


# 9. Zip và download


In [ ]:
!zip -r /content/batch_output.zip /content/batch_output
from google.colab import files
files.download('/content/batch_output.zip')
print('Tóm tắt: BatchChapter hoàn tất')
